In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/"
conf   = pd.read_csv(BASE + "csse_covid_19_time_series/time_series_covid19_confirmed_global.csv")
deaths = pd.read_csv(BASE + "csse_covid_19_time_series/time_series_covid19_deaths_global.csv")
# keep_default_na=False so Namibia's code "NA" is not read as missing
lookup = pd.read_csv(BASE + "UID_ISO_FIPS_LookUp_Table.csv", keep_default_na=False, na_values=[""])

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
conf.to_csv("data/raw/confirmed_global.csv", index=False)
deaths.to_csv("data/raw/deaths_global.csv", index=False)
lookup.to_csv("data/raw/uid_lookup.csv", index=False)

def to_country_daily(df, name):
    long = df.melt(id_vars=["Province/State", "Country/Region", "Lat", "Long"],
                   var_name="date", value_name=name)
    long["date"] = pd.to_datetime(long["date"], format="%m/%d/%y")
    # add provinces up to country level FIRST
    return long.groupby(["Country/Region", "date"], as_index=False)[name].sum()

daily = to_country_daily(conf, "confirmed").merge(
    to_country_daily(deaths, "deaths"), on=["Country/Region", "date"])
daily = daily.rename(columns={"Country/Region": "country"})

# check BEFORE dropping anything: world totals
world = daily.groupby("date")[["confirmed", "deaths"]].sum().iloc[-1]
print(world)   # expect about 676 million cases and 6.9 million deaths
assert 6.6e8 < world["confirmed"] < 6.9e8 and 6.8e6 < world["deaths"] < 7.0e6

# remove entries that are not countries
daily = daily[~daily["country"].isin(
    ["Diamond Princess", "MS Zaandam", "Summer Olympics 2020", "Winter Olympics 2022"])]

# population + ISO code from JHU's own lookup table
lk = lookup[lookup["Admin2"].isna()]
country_rows = lk[lk["Province_State"].isna()]
pop = (country_rows.groupby("Country_Region")["Population"].sum(min_count=1)
       .combine_first(lk[lk["Province_State"].notna()].groupby("Country_Region")["Population"].sum(min_count=1)))
iso2 = country_rows.groupby("Country_Region")["iso2"].first()
daily["population"] = daily["country"].map(pop)
print("No population for:", sorted(daily.loc[daily["population"].isna(), "country"].unique()))

# KPIs per country
daily = daily.sort_values(["country", "date"]).reset_index(drop=True)
for total, new in [("confirmed", "new_cases"), ("deaths", "new_deaths")]:
    daily[new] = daily.groupby("country")[total].diff().fillna(daily[total]).clip(lower=0)  # clip = data corrections
daily["cases_7d_avg"] = daily.groupby("country")["new_cases"].transform(lambda s: s.rolling(7).mean())
daily["deaths_7d_avg"] = daily.groupby("country")["new_deaths"].transform(lambda s: s.rolling(7).mean())
daily["cfr_pct"] = daily["deaths"] / daily["confirmed"].where(daily["confirmed"] > 0) * 100   # blank, not 0, when no cases
daily["cases_7d_avg_per_million"] = daily["cases_7d_avg"] / daily["population"] * 1e6

# summary table: one row per country
last = daily["date"].max()
summary = daily.loc[daily["date"] == last, ["country", "population", "confirmed", "deaths"]].copy()
summary["iso2"] = summary["country"].map(iso2)
summary["cfr_pct"] = summary["deaths"] / summary["confirmed"].where(summary["confirmed"] > 0) * 100
summary["cases_per_million"] = summary["confirmed"] / summary["population"] * 1e6
summary["deaths_per_million"] = summary["deaths"] / summary["population"] * 1e6
peak = daily.loc[daily.groupby("country")["cases_7d_avg"].idxmax(), ["country", "date", "cases_7d_avg"]]
summary = summary.merge(peak.rename(columns={"date": "peak_date", "cases_7d_avg": "peak_7d_avg"}), on="country")

daily.round(2).to_csv("data/processed/covid_country_daily.csv", index=False)
summary.round(2).to_csv("data/processed/covid_country_summary.csv", index=False)
print(daily.shape, summary.shape, last)

confirmed    676570149
deaths         6881802
Name: 2023-03-09 00:00:00, dtype: int64
No population for: ['Antarctica']
(225171, 11) (197, 10) 2023-03-09 00:00:00


In [ ]:
!pip -q install duckdb
!mkdir -p sql

In [ ]:
%%writefile sql/covid_queries.sql
-- Create tables from the processed CSV files
CREATE OR REPLACE TABLE daily AS SELECT * FROM read_csv_auto('data/processed/covid_country_daily.csv', sample_size=-1);
CREATE OR REPLACE TABLE summary AS SELECT * FROM read_csv_auto('data/processed/covid_country_summary.csv', sample_size=-1);

-- 1. Sanity check: world totals (JHU final figures are about 676.6 million cases and 6.88 million deaths)
SELECT SUM(confirmed) AS world_cases, SUM(deaths) AS world_deaths,
       ROUND(SUM(deaths) * 100.0 / SUM(confirmed), 2) AS world_cfr_pct
FROM summary;

-- 2. Top 10 countries by total cases, with share of the world total
SELECT country, confirmed AS total_cases, deaths AS total_deaths, cfr_pct,
       ROUND(confirmed * 100.0 / SUM(confirmed) OVER (), 1) AS share_of_world_pct
FROM summary
ORDER BY confirmed DESC
LIMIT 10;

-- 3. Top 10 countries by cases per million (only countries with 1 million+ people)
SELECT country, ROUND(cases_per_million) AS cases_per_million,
       ROUND(deaths_per_million) AS deaths_per_million, cfr_pct
FROM summary
WHERE population >= 1000000
ORDER BY cases_per_million DESC
LIMIT 10;

-- 4. How the ranking changes after adjusting for population (countries with 1 million+ people)
SELECT country, confirmed AS total_cases,
       RANK() OVER (ORDER BY confirmed DESC) AS rank_total_cases,
       RANK() OVER (ORDER BY cases_per_million DESC) AS rank_per_million
FROM summary
WHERE population >= 1000000
ORDER BY rank_total_cases
LIMIT 10;

-- 5. Each country's worst month for new cases, then count countries per month
WITH monthly AS (
    SELECT country, date_trunc('month', date) AS month_start, SUM(new_cases) AS cases
    FROM daily
    GROUP BY country, month_start
), ranked AS (
    SELECT country, month_start, cases,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY cases DESC) AS rn
    FROM monthly
)
SELECT month_start AS worst_month, COUNT(*) AS countries
FROM ranked
WHERE rn = 1
GROUP BY month_start
ORDER BY countries DESC
LIMIT 5;

-- 6. Global fatality rate by year (deaths in the year divided by cases in the year)
SELECT EXTRACT(YEAR FROM date) AS yr, SUM(new_cases) AS cases, SUM(new_deaths) AS deaths,
       ROUND(SUM(new_deaths) * 100.0 / SUM(new_cases), 2) AS cfr_pct
FROM daily
GROUP BY yr
ORDER BY yr;

-- 7. Cross-check: 7-day average calculated in SQL vs the Python column (difference should be below 0.01)
WITH w AS (
    SELECT country, date,
           AVG(new_cases) OVER (PARTITION BY country ORDER BY date
                                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS sql_7d_avg,
           cases_7d_avg AS python_7d_avg
    FROM daily
)
SELECT MAX(ABS(sql_7d_avg - python_7d_avg)) AS max_difference
FROM w
WHERE python_7d_avg IS NOT NULL;

Overwriting sql/covid_queries.sql


In [ ]:
import duckdb
from IPython.display import display

con = duckdb.connect()
for stmt in open("sql/covid_queries.sql").read().split(";"):
    if stmt.strip():
        result = con.sql(stmt)
        if result is not None:                 # CREATE TABLE gives no table back
            print(stmt.strip().splitlines()[0])
            display(result.df())

-- 1. Sanity check: world totals (JHU final figures are about 676.6 million cases and 6.88 million deaths)


,world_cases,world_deaths,world_cfr_pct
0,676568028.0,6881787.0,1.02


-- 2. Top 10 countries by total cases, with share of the world total


,country,total_cases,total_deaths,cfr_pct,share_of_world_pct
0,US,103802702,1123836,1.08,15.3
1,India,44690738,530779,1.19,6.6
2,France,39866718,166176,0.42,5.9
3,Germany,38249060,168935,0.44,5.7
4,Brazil,37076053,699276,1.89,5.5
5,Japan,33320438,72997,0.22,4.9
6,"Korea, South",30615522,34093,0.11,4.5
7,Italy,25603510,188322,0.74,3.8
8,United Kingdom,24658705,220721,0.90,3.6
9,Russia,22075858,388478,1.76,3.3


-- 3. Top 10 countries by cases per million (only countries with 1 million+ people)


,country,cases_per_million,deaths_per_million,cfr_pct
0,Austria,661879.0,2439.0,0.37
1,Slovenia,640573.0,3405.0,0.53
2,France,610986.0,2547.0,0.42
3,"Korea, South",597153.0,665.0,0.11
4,Denmark,591213.0,1430.0,0.24
5,Israel,555000.0,1424.0,0.26
6,Portugal,546301.0,2576.0,0.47
7,Cyprus,538932.0,1102.0,0.20
8,Greece,532328.0,3337.0,0.63
9,Latvia,517577.0,3324.0,0.64


... (4 outputs left out)

-- 6. Global fatality rate by year (deaths in the year divided by cases in the year)


,yr,cases,deaths,cfr_pct
0,2020,83896261.0,1905067.0,2.27
1,2021,205448835.0,3568659.0,1.74
2,2022,371951091.0,1227029.0,0.33
3,2023,16123170.0,188711.0,1.17


-- 7. Cross-check: 7-day average calculated in SQL vs the Python column (difference should be below 0.01)


,max_difference
0,0.004286
